# Informe estándar por equipo — Grupo A 2026

Genera el **mismo informe completo que el de Chaco** (19 gráficos + tablas + `.tex`
compilado) para **cualquier equipo del grupo**, sin nada hardcodeado de Chaco.
Pensado para enviar a cada club como muestra del servicio.

**Cómo usarlo:** cambiá `TEAM_SLUG` (y opcionalmente `DESTINATARIO`), corré todo, y
el PDF queda en `outputs/temporadas/2026/global_grupo_a/equipos/<slug>/`.

In [1]:
import os, sys, subprocess
%load_ext autoreload
%autoreload 2
if os.path.basename(os.getcwd()) in ('primera_nacional', 'liga_profesional'):
    os.chdir('..')
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
PROJECT_ROOT = os.getcwd(); sys.path.insert(0, PROJECT_ROOT)
import config
from src.analysis import liga
from src.reports.liga import build_team_report_tex
from src.visualization import liga as vliga
from IPython.display import Image, display
def show(p):
    if p and os.path.exists(p): display(Image(filename=p))
print('Grupo A:', ', '.join(config.group_a_teams('2026')[s]['name'] for s in list(config.group_a_teams('2026'))[:6]), '...')

Grupo A: Chaco For Ever, Acassuso, All Boys, Almirante Brown, Central Norte de Salta, Ciudad de Bolívar ...


## 1. Configuración — elegí el equipo

In [2]:
# ============================================================
# CONFIGURAR ACÁ
SEASON       = '2026'
TEAM_SLUG    = 'san_telmo'          # slug del equipo (ver lista abajo)
DESTINATARIO = None                 # None = 'Cuerpo Técnico y Dirigencia de <equipo>'
# ============================================================
print('Equipos disponibles (slug -> nombre):')
for s, info in config.group_a_teams(SEASON).items():
    print(f'  {s:26} {info["name"]}')

OUT_DIR = config.team_report_dir(TEAM_SLUG, SEASON)
os.makedirs(OUT_DIR, exist_ok=True)
print('\nSalida:', OUT_DIR)

Equipos disponibles (slug -> nombre):
  chaco_for_ever             Chaco For Ever
  acassuso                   Acassuso
  all_boys                   All Boys
  almirante_brown            Almirante Brown
  central_norte              Central Norte de Salta
  ciudad_de_bolivar          Ciudad de Bolívar
  colon                      Club Atlético Colón
  defensores_de_belgrano     Defensores de Belgrano
  deportivo_madryn           Deportivo Madryn
  deportivo_moron            Deportivo Morón
  estudiantes_caseros        Estudiantes de Buenos Aires
  ferro_carril_oeste         Club Ferro Carril Oeste
  gimnasia_de_jujuy          Gimnasia Jujuy
  godoy_cruz                 Godoy Cruz
  los_andes                  Los Andes
  mitre                      CA Mitre
  racing_de_cordoba          Racing de Córdoba
  san_miguel                 San Miguel
  san_telmo                  San Telmo

Salida: C:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\outputs\temporadas\2026\global_grupo_a\eq

## 2. Generar los 19 gráficos + tablas del equipo

In [3]:
res = liga.generate_team_graphics(TEAM_SLUG, SEASON,
                                  plots_dir=os.path.join(OUT_DIR, 'graficos'))
st = res['stats']
print(f"\n{res['name']}: {res['n_matches']} PJ | "
      f"{st['victorias']}V-{st['empates']}E-{st['derrotas']}D ({st['puntos']} pts) | "
      f"GF {st['goles_favor']} / GC {st['goles_contra']}")
print('Gráficos generados:', len(res['graphics']))

  ✔ puntos_acumulados
  ✔ xg_temporada


  ✔ top_minutos
  ✔ top_rating


  ✔ shotmap_temporada


  ✔ red_minutos_compartidos


  ✔ tiros_concedidos_temporada
  ✔ forma_reciente


...

  ✔ amenaza_por_jugada
  ✔ xg_por_estado_marcador


  ✔ toques_ultimo_tercio


  ✔ markov_marcador
  ✔ nucleo_tactico


  ✔ entropia_tiros


  ✔ riesgo_defensivo_cox

San Telmo: 20 PJ | 4V-8E-8D (20 pts) | GF 16 / GC 21
Gráficos generados: 19


In [ ]:
# Estadísticas que los 19 gráficos no muestran, a favor y en contra
# (promedio por partido): npxG, xGOT, cabeza, pelota parada, palos, penales, km.
import pandas as pd
from src.analysis.pre_match import team_two_way_profile
_base, _mks, _ = liga.team_context(TEAM_SLUG, SEASON)
prof = team_two_way_profile(_mks, _base)
filas = [('npxG', 'npxg'), ('xGOT', 'xgot'), ('Remates de cabeza', 'headers'),
         ('Remates de pelota parada', 'setpiece_shots'), ('Tiros al palo', 'woodwork'),
         ('Penales', 'penalties'), ('Km recorridos', 'km'), ('Sprints', 'sprints')]
equipo_extra = pd.DataFrame([{'metrica': lbl, 'a_favor': prof.get(f'{k}_for'),
                              'en_contra': prof.get(f'{k}_against')} for lbl, k in filas])
equipo_extra.to_csv(os.path.join(OUT_DIR, 'estadisticas_extra_equipo.csv'), index=False)
display(equipo_extra)
print(f"{prof['n_shotmap']} partidos con shotmap · {prof['n_tracking']} con km/sprints")

## 3. Armar y compilar el informe `.tex` (marca de la consultora + destinatario)

In [4]:
tex = build_team_report_tex(res, destinatario=DESTINATARIO)
tex_path = os.path.join(OUT_DIR, f'Informe_{TEAM_SLUG}.tex')
with open(tex_path, 'w', encoding='utf-8') as f:
    f.write(tex)

LUALATEX = r'C:/Users/luart/AppData/Local/Programs/MiKTeX/miktex/bin/x64/lualatex.exe'
for _ in range(2):  # dos pasadas por \pageref{LastPage}
    subprocess.run([LUALATEX, '-interaction=nonstopmode', '-halt-on-error',
                    os.path.basename(tex_path)], cwd=OUT_DIR,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
pdf = tex_path.replace('.tex', '.pdf')
print('PDF:', pdf, '| existe:', os.path.exists(pdf))

PDF: C:\Users\luart\OneDrive\Escritorio\Chaco For Ever Analisis\outputs\temporadas\2026\global_grupo_a\equipos\san_telmo\Informe_san_telmo.pdf | existe: True


## Listo
El `.tex` tiene espacios de narrativa (`\slot`) para completar a mano antes de enviarlo.
Para otro equipo: cambiá `TEAM_SLUG` arriba y corré de nuevo.